In [ ]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok -q

In [ ]:
%%writefile app.py
import streamlit as st
import sqlite3
import bcrypt
import jwt
import re
import secrets
import time
import smtplib
import os
import hmac
import base64

from contextlib import contextmanager
from email.message import EmailMessage
from datetime import datetime, timedelta, timezone


# ---------------- CONFIGURATION ----------------
st.set_page_config(
    page_title="Maritime Freight Pricing and Route Optimization",
    layout="centered",
    initial_sidebar_state="collapsed"
)

os.environ.setdefault("JWT_SECRET", secrets.token_hex(32))

ADMIN_USERNAME = os.environ.get("ADMIN_USERNAME", "admin")
ADMIN_PASSWORD = os.environ.get("ADMIN_PASSWORD", "admin@123")

DB = "users.db"

PROJECT_NAME = "Maritime Freight Pricing and Route Optimization"
PROJECT_LINK = os.environ.get(
    "PROJECT_LINK",
    "https://github.com/YOUR-USERNAME/YOUR-REPO"
)

QUESTIONS = [
    "What was the name of your first school?",
    "What is your childhood best friend's name?",
    "What city were you born in?"
]


# ---------------- BACKGROUND + STYLING ----------------
BG_B64 = "..."


def load_background():
    return BG_B64


CSS = """
<style>
header[data-testid="stHeader"], [data-testid="stToolbar"],
[data-testid="stSidebar"], [data-testid="collapsedControl"],
[data-testid="stSidebarCollapsedControl"], footer {
    display: none !important;
}

.stApp, [data-testid="stAppViewContainer"] {
    background-color: #0a3a78 !important;
    background-image: __BG__ !important;
    background-size: 100% auto !important;
    background-repeat: no-repeat !important;
    background-position: top center !important;
}
[data-testid="stMain"], [data-testid="stBottom"],
[data-testid="stBottomBlockContainer"] {
    background: transparent !important;
}

/* The white login card */
.block-container {
    max-width: max(49.4vw, 340px) !important;
    margin: 35.7vw auto 3rem auto !important;
    padding: 3rem 2rem 1.5rem 2rem !important;
    background: #ffffff;
    border-radius: 10px;
    box-shadow: 0 8px 30px rgba(0, 20, 60, 0.35);
    color: #0b2a4a;
}

.block-container, .block-container p, .block-container label,
.block-container span, .block-container li, .block-container h1,
.block-container h2, .block-container h3 {
    color: #0b2a4a;
}

.anchor-icon {
    width: 56px; height: 56px; border-radius: 50%;
    background: #e3eefb; border: 4px solid #ffffff;
    box-shadow: 0 2px 8px rgba(0, 0, 0, 0.15);
    margin: -76px auto 6px auto;
    display: flex; align-items: center; justify-content: center;
    font-size: 26px; color: #0b3d66;
}
.card-title {
    text-align: center; font-size: 1.6rem; font-weight: 700;
    color: #0b3d66; margin: 0; line-height: 1.2;
}
.card-sub {
    text-align: center; font-size: 0.9rem; color: #5b6b7d;
    margin: 4px 0 10px 0;
}
.or-line {
    display: flex; align-items: center; gap: 10px;
    color: #8a97a6; font-size: 0.8rem; margin: 2px 0;
}
.or-line::before, .or-line::after {
    content: ""; flex: 1; height: 1px; background: #d9e2ec;
}

/* Inputs */
div[data-baseweb="input"], div[data-baseweb="select"] > div {
    background: #f5f9fd !important;
    border: 1px solid #cfe0f0 !important;
    border-radius: 8px !important;
}
div[data-baseweb="input"] input {
    background: transparent !important;
    color: #0b2a4a !important;
}
div[data-baseweb="input"] input::placeholder { color: #8fa3b8 !important; }

/* Password eye icon: white icon on a teal chip */
div[data-baseweb="input"] > div {
    background: transparent !important;
    background-color: transparent !important;
}
div[data-baseweb="input"] button,
[data-testid="stTextInputRootElement"] button {
    background: #0e7c8f !important;
    background-color: #0e7c8f !important;
    border: none !important;
    border-radius: 6px !important;
    box-shadow: none !important;
    color: #ffffff !important;
}
div[data-baseweb="input"] button svg,
div[data-baseweb="input"] button svg path {
    fill: #ffffff !important;
    color: #ffffff !important;
}
div[data-baseweb="input"] button:hover,
div[data-baseweb="input"] button:focus,
div[data-baseweb="input"] button:active {
    background: #0b6678 !important;
    background-color: #0b6678 !important;
}

/* Forms: no extra border */
[data-testid="stForm"] {
    border: none !important;
    padding: 0 !important;
}

/* Buttons: full width */
.stButton, [data-testid="stFormSubmitButton"],
[data-testid="stElementContainer"]:has(button),
[data-testid="stElementContainer"]:has(button) > div {
    width: 100% !important;
}
.stButton > button, [data-testid="stFormSubmitButton"] > button {
    width: 100% !important;
}

/* Buttons */
.stButton > button, [data-testid="stFormSubmitButton"] > button {
    width: 100%;
    min-height: 2.8rem;
    border-radius: 8px;
    font-weight: 600;
}
[data-testid="stFormSubmitButton"] > button {
    background: #0e7c8f !important;
    border: 1px solid #0e7c8f !important;
    color: #ffffff !important;
}
[data-testid="stFormSubmitButton"] > button p { color: #ffffff !important; }
[data-testid="stFormSubmitButton"] > button:hover {
    background: #0b6678 !important;
}
.stButton > button {
    background: #ffffff !important;
    border: 1px solid #1f6fb5 !important;
    color: #1f6fb5 !important;
}
.stButton > button p { color: #1f6fb5 !important; }
.stButton > button:hover { background: #eef5fc !important; }
</style>
"""

_bg = load_background()
_bg_css = (
    f"url('data:image/jpeg;base64,{_bg}')"
    if _bg
    else "linear-gradient(180deg, #0b4a8f 0%, #062b58 100%)"
)
st.markdown(CSS.replace("__BG__", _bg_css), unsafe_allow_html=True)


def card_header(title, subtitle):
    st.markdown(
        f"""
        <div class="anchor-icon">&#9875;</div>
        <p class="card-title">{title}</p>
        <p class="card-sub">{subtitle}</p>
        """,
        unsafe_allow_html=True
    )


# ---------------- HELPERS ----------------
def get_secret(name):
    value = os.environ.get(name)
    if not value:
        raise ValueError(f"{name} is not set in the environment.")
    return value


@contextmanager
def connect():
    conn = sqlite3.connect(DB, timeout=10)
    conn.row_factory = sqlite3.Row

    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


CREATE_USERS_SQL = """
    CREATE TABLE IF NOT EXISTS users (
        username TEXT PRIMARY KEY,
        email TEXT NOT NULL,
        password_hash BLOB NOT NULL,
        question TEXT NOT NULL,
        answer_hash BLOB NOT NULL
    )
"""


def initialize_database():
    with connect() as conn:
        conn.execute(CREATE_USERS_SQL)

        email_is_unique = False

        for idx in conn.execute("PRAGMA index_list(users)").fetchall():
            if idx["unique"] and idx["origin"] == "u":
                cols = [
                    c["name"]
                    for c in conn.execute(
                        f"PRAGMA index_info('{idx['name']}')"
                    ).fetchall()
                ]
                if cols == ["email"]:
                    email_is_unique = True

        if email_is_unique:
            conn.execute("ALTER TABLE users RENAME TO users_old")
            conn.execute(CREATE_USERS_SQL)
            conn.execute(
                """
                INSERT INTO users
                (username, email, password_hash, question, answer_hash)
                SELECT username, email, password_hash,
                       question, answer_hash
                FROM users_old
                """
            )
            conn.execute("DROP TABLE users_old")


initialize_database()


def hash_text(value):
    return bcrypt.hashpw(value.encode("utf-8"), bcrypt.gensalt())


def verify_hash(value, hashed):
    try:
        if isinstance(hashed, str):
            hashed = hashed.encode("utf-8")
        return bcrypt.checkpw(value.encode("utf-8"), hashed)
    except (ValueError, TypeError, AttributeError):
        return False


def valid_email(email):
    pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    return re.fullmatch(pattern, email.strip()) is not None


def valid_password(password):
    return bool(
        len(password) >= 8
        and re.search(r"[A-Z]", password)
        and re.search(r"[a-z]", password)
        and re.search(r"[0-9]", password)
        and re.search(r"[^A-Za-z0-9]", password)
    )


def create_jwt(username, hours=2):
    now = datetime.now(timezone.utc)

    payload = {
        "sub": username,
        "iat": now,
        "exp": now + timedelta(hours=hours)
    }

    return jwt.encode(payload, get_secret("JWT_SECRET"), algorithm="HS256")


def check_jwt(token):
    if not token:
        return None

    try:
        data = jwt.decode(
            token,
            get_secret("JWT_SECRET"),
            algorithms=["HS256"]
        )
        return data["sub"]
    except (jwt.PyJWTError, ValueError, KeyError, TypeError):
        return None


def send_otp(email, otp):
    sender = get_secret("EMAIL_ADDRESS")
    password = get_secret("EMAIL_PASSWORD")

    message = EmailMessage()
    message["Subject"] = f"{PROJECT_NAME} - Verification Code"
    message["From"] = f"{PROJECT_NAME} <{sender}>"
    message["To"] = email

    message.set_content(
        f"{PROJECT_NAME}\n"
        f"Verification Code\n\n"
        f"Your verification code is: {otp}\n"
        f"It expires in 5 minutes.\n\n"
        f"Project link: {PROJECT_LINK}\n\n"
        f"If you did not request this, please ignore this email."
    )

    message.add_alternative(f"""
    <html>
      <body style="font-family:Arial;background:#f4f6f8;padding:20px;">
        <div style="max-width:480px;margin:auto;background:#fff;
                    border-radius:8px;padding:24px;text-align:center;">
          <h2 style="color:#0b3d66;">{PROJECT_NAME}</h2>
          <h3>Verification Code</h3>
          <p>Use this code to reset your password:</p>
          <p style="font-size:32px;letter-spacing:8px;font-weight:bold;
                    color:#0b3d66;">{otp}</p>
          <p style="color:#777;font-size:13px;">
            This code expires in 5 minutes.
          </p>
          <a href="{PROJECT_LINK}"
             style="display:inline-block;margin-top:12px;padding:10px 20px;
                    background:#0b3d66;color:white;text-decoration:none;
                    border-radius:6px;">
            Open Project
          </a>
        </div>
      </body>
    </html>
    """, subtype="html")

    with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as server:
        server.login(sender, password)
        server.send_message(message)


# ---------------- SESSION STATE ----------------
defaults = {
    "token": None,
    "otp": None,
    "admin_logged": False,
    "reset_user": None,
    "page": "Login",
}

for key, default in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = default


def go(page_name):
    st.session_state.page = page_name


def back_to_login_button():
    st.button("Back to Login", on_click=go, args=("Login",),
              key=f"back_{st.session_state.page}")


# ---------------- LOGGED-IN DASHBOARD ----------------
logged_user = check_jwt(st.session_state.token)

if logged_user:
    card_header("User Dashboard", f"Welcome, {logged_user}!")
    st.success(f"Welcome, {logged_user}!")
    st.write("You are logged in with a valid JWT session.")

    if st.button("Logout"):
        st.session_state.token = None
        st.rerun()

    st.stop()

elif st.session_state.token:
    st.session_state.token = None


page = st.session_state.page


# ---------------- LOGIN ----------------
if page == "Login":
    card_header("Welcome Back",
                "Sign in to access the maritime platform")

    with st.form("login"):
        identity = st.text_input(
            "Username or Email",
            placeholder="Username or Email",
            label_visibility="collapsed"
        ).strip()

        password = st.text_input(
            "Password",
            type="password",
            placeholder="Password",
            label_visibility="collapsed"
        )

        submit = st.form_submit_button("Login")

    if submit:
        if not identity or not password:
            st.error("All fields are mandatory.")

        else:
            rows = None
            try:
                with connect() as conn:
                    rows = conn.execute(
                        """
                        SELECT *
                        FROM users
                        WHERE username = ? OR email = ?
                        """,
                        (identity, identity.lower())
                    ).fetchall()

            except sqlite3.Error as e:
                st.error(f"Database error: {e}")

            if rows is not None:
                user = next(
                    (
                        u for u in rows
                        if verify_hash(password, u["password_hash"])
                    ),
                    None
                )

                if user:
                    try:
                        st.session_state.token = create_jwt(
                            user["username"],
                            hours=2
                        )

                    except ValueError as e:
                        st.error(f"Server configuration error: {e}")

                    else:
                        st.rerun()

                else:
                    st.error("Invalid username/email or password.")

    st.button("Forgot password?", on_click=go,
              args=("Forgot Password",), key="to_forgot")

    st.markdown('<div class="or-line">OR</div>', unsafe_allow_html=True)

    st.button("Create New Account", on_click=go,
              args=("Signup",), key="to_signup")

    st.button("Admin Login", on_click=go,
              args=("Admin Dashboard",), key="to_admin")


# ---------------- SIGNUP ----------------
elif page == "Signup":
    card_header("Create Account", "Join the maritime platform")

    with st.form("signup"):
        username = st.text_input("Username").strip()
        email = st.text_input("Email").strip().lower()
        password = st.text_input("Password", type="password")
        confirm = st.text_input("Confirm Password", type="password")
        question = st.selectbox("Security Question", QUESTIONS)
        answer = st.text_input("Security Answer", type="password")

        submit = st.form_submit_button("Signup")

    if submit:
        if not all([username, email, password, confirm, answer.strip()]):
            st.error("All fields are mandatory.")

        elif not re.fullmatch(r"[A-Za-z][A-Za-z0-9-]{2,29}", username):
            st.error(
                "Username must start with a letter and contain "
                "3-30 letters, numbers or hyphens."
            )

        elif not valid_email(email):
            st.error("Invalid email format.")

        elif not valid_password(password):
            st.error(
                "Password needs 8+ characters, uppercase, "
                "lowercase, number and special symbol."
            )

        elif password != confirm:
            st.error("Passwords do not match.")

        else:
            try:
                with connect() as conn:
                    conn.execute(
                        """
                        INSERT INTO users
                        (username, email, password_hash,
                         question, answer_hash)
                        VALUES (?, ?, ?, ?, ?)
                        """,
                        (
                            username,
                            email,
                            hash_text(password),
                            question,
                            hash_text(answer.strip().lower())
                        )
                    )

                st.success("Signup successful. Please log in.")

            except sqlite3.IntegrityError:
                st.error("Username already exists.")

            except sqlite3.Error as e:
                st.error(f"Database error: {e}")

    back_to_login_button()


# ---------------- FORGOT PASSWORD ----------------
elif page == "Forgot Password":
    card_header("Forgot Password", "Recover access to your account")

    method = st.radio(
        "Recovery Method",
        ["Security Question", "Email OTP"]
    )

    # ----- Security Question -----
    if method == "Security Question":
        lookup_name = st.text_input("Username", key="sq_username").strip()

        if lookup_name:
            try:
                with connect() as conn:
                    row = conn.execute(
                        "SELECT question FROM users WHERE username=?",
                        (lookup_name,)
                    ).fetchone()

                if row:
                    st.info(f"Security question: {row['question']}")
                else:
                    st.warning("Username not found.")

            except sqlite3.Error as e:
                st.error(f"Database error: {e}")

        with st.form("security_reset"):
            answer = st.text_input("Security Answer", type="password")
            new_password = st.text_input("New Password", type="password")
            confirm = st.text_input("Confirm New Password", type="password")

            submit = st.form_submit_button("Reset Password")

        if submit:
            if not all([lookup_name, answer.strip(), new_password, confirm]):
                st.error("All fields are mandatory.")

            elif not valid_password(new_password):
                st.error("New password does not meet the password rules.")

            elif new_password != confirm:
                st.error("Passwords do not match.")

            else:
                try:
                    with connect() as conn:
                        user = conn.execute(
                            "SELECT * FROM users WHERE username=?",
                            (lookup_name,)
                        ).fetchone()

                    if user and verify_hash(
                        answer.strip().lower(), user["answer_hash"]
                    ):
                        with connect() as conn:
                            conn.execute(
                                """
                                UPDATE users
                                SET password_hash=?
                                WHERE username=?
                                """,
                                (hash_text(new_password), lookup_name)
                            )

                        st.success(
                            "Password reset successful. Please log in."
                        )

                    else:
                        st.error(
                            "Username or security answer is incorrect."
                        )

                except sqlite3.Error as e:
                    st.error(f"Database error: {e}")

    # ----- Email OTP -----
    else:
        with st.form("send_otp"):
            email = st.text_input("Registered Email").strip().lower()
            send_clicked = st.form_submit_button("Send OTP")

        if send_clicked:
            if not email or not valid_email(email):
                st.error("Enter a valid registered email.")

            else:
                try:
                    with connect() as conn:
                        user = conn.execute(
                            """
                            SELECT username, email
                            FROM users
                            WHERE email=?
                            """,
                            (email,)
                        ).fetchone()

                    if user:
                        code = f"{secrets.randbelow(1000000):06d}"

                        send_otp(email, code)

                        st.session_state.otp = {
                            "code_hash": hash_text(code),
                            "username": user["username"],
                            "email": email,
                            "expires": time.time() + 300,
                            "attempts": 0,
                            "verified": False
                        }

                        st.success("OTP sent to your email.")

                    else:
                        st.error("Email not found.")

                except ValueError as e:
                    st.error(f"Server configuration error: {e}")

                except smtplib.SMTPAuthenticationError:
                    st.error(
                        "Email login failed. Use a Gmail App Password "
                        "for EMAIL_PASSWORD."
                    )

                except Exception as e:
                    st.error(f"Could not send OTP: {e}")

        state = st.session_state.otp

        if state and not state.get("verified"):
            with st.form("verify_otp"):
                entered = st.text_input(
                    "Enter 6-digit OTP", max_chars=6
                ).strip()

                verify_clicked = st.form_submit_button("Verify OTP")

            if verify_clicked:
                if time.time() > state["expires"]:
                    st.session_state.otp = None
                    st.error("OTP expired. Request a new OTP.")

                elif state["attempts"] >= 5:
                    st.session_state.otp = None
                    st.error("Too many attempts. Request a new OTP.")

                elif verify_hash(entered, state["code_hash"]):
                    state["verified"] = True
                    st.session_state.otp = state
                    st.rerun()

                else:
                    state["attempts"] += 1
                    st.session_state.otp = state
                    st.error("Incorrect OTP.")

        state = st.session_state.otp

        if state and state.get("verified"):
            if time.time() > state["expires"]:
                st.session_state.otp = None
                st.error("Session expired. Request a new OTP.")

            else:
                st.success("OTP verified. Set your new password.")

                with st.form("new_password"):
                    new_password = st.text_input(
                        "New Password", type="password"
                    )
                    confirm = st.text_input(
                        "Confirm New Password", type="password"
                    )

                    save = st.form_submit_button("Save Password")

                if save:
                    if not new_password or not confirm:
                        st.error("Both fields are mandatory.")

                    elif not valid_password(new_password):
                        st.error("Password does not meet the required rules.")

                    elif new_password != confirm:
                        st.error("Passwords do not match.")

                    else:
                        try:
                            with connect() as conn:
                                conn.execute(
                                    """
                                    UPDATE users
                                    SET password_hash=?
                                    WHERE username=?
                                    """,
                                    (
                                        hash_text(new_password),
                                        state["username"]
                                    )
                                )

                            st.session_state.otp = None

                            st.success(
                                "Password reset successful. Please log in."
                            )

                        except sqlite3.Error as e:
                            st.error(f"Database error: {e}")

    back_to_login_button()


# ---------------- ADMIN DASHBOARD ----------------
elif page == "Admin Dashboard":
    card_header("Admin", "Administrator access")

    if not st.session_state.admin_logged:
        with st.form("admin_login"):
            admin_user = st.text_input("Admin Username")
            admin_password = st.text_input("Admin Password", type="password")

            submit = st.form_submit_button("Login as Admin")

        if submit:
            if (
                hmac.compare_digest(
                    admin_user.encode("utf-8"),
                    ADMIN_USERNAME.encode("utf-8")
                )
                and hmac.compare_digest(
                    admin_password.encode("utf-8"),
                    ADMIN_PASSWORD.encode("utf-8")
                )
            ):
                st.session_state.admin_logged = True
                st.rerun()

            else:
                st.error("Invalid admin credentials.")

    if st.session_state.admin_logged:

        if st.session_state.get("deleted_msg"):
            st.success(st.session_state.pop("deleted_msg"))

        st.subheader("Registered Users")

        try:
            with connect() as conn:
                users = conn.execute(
                    """
                    SELECT username, email
                    FROM users
                    ORDER BY username
                    """
                ).fetchall()

            st.dataframe(
                [
                    {"Username": u["username"], "Email": u["email"]}
                    for u in users
                ]
            )

            if users:
                st.subheader("Delete User")

                usernames = [u["username"] for u in users]

                user_to_delete = st.selectbox(
                    "Select user to delete", usernames
                )

                confirm_delete = st.checkbox(
                    f"Yes, delete '{user_to_delete}' permanently"
                )

                if st.button("Delete User"):
                    if not confirm_delete:
                        st.warning("Please tick the confirmation box.")
                    else:
                        with connect() as conn:
                            conn.execute(
                                "DELETE FROM users WHERE username = ?",
                                (user_to_delete,)
                            )
                        st.session_state.deleted_msg = (
                            f"User '{user_to_delete}' deleted."
                        )
                        st.rerun()

        except sqlite3.Error as e:
            st.error(f"Database error: {e}")

        if st.button("Admin Logout"):
            st.session_state.admin_logged = False
            st.rerun()

    back_to_login_button()


Writing app.py


In [ ]:
import os, time, socket, subprocess
from google.colab import userdata, files
from pyngrok import ngrok, conf

# 1) Load secrets AND pass them to Streamlit (it runs as a separate process)
for name in ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD"]:
    value = userdata.get(name)
    if not value:
        raise ValueError(f"Missing secret: {name}")
    os.environ[name] = value

# 2) Make sure files exist
if not os.path.exists("app.py"):
    raise FileNotFoundError("app.py not found - run the previous cell (%%writefile app.py) first.")
if not os.path.exists("background.jpg"):
    print("Select background.jpg to upload...")
    files.upload()

# 3) Clean up previous runs
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit run"])
time.sleep(2)

conf.get_default().auth_token = os.environ["NGROK_AUTHTOKEN"]

# 4) Start Streamlit and log its output
log = open("streamlit.log", "w")
process = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true"],
    stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy()
)

# 5) Wait until Streamlit is really listening (fail loudly if it crashes)
for _ in range(60):
    if process.poll() is not None:
        print(open("streamlit.log").read())
        raise RuntimeError("Streamlit crashed - see the log above.")
    try:
        socket.create_connection(("127.0.0.1", 8501), timeout=1).close()
        break
    except OSError:
        time.sleep(1)
else:
    print(open("streamlit.log").read())
    raise RuntimeError("Streamlit did not start in 60 seconds.")

# 6) Connect using port 8501 to avoid invalid URL parsing issues in pyngrok
tunnel = ngrok.connect(8501)
print("Open your project here:")
print(tunnel.public_url)


Select background.jpg to upload...


Saving WhatsApp Image 2026-11-01 at 10.23.28 AM.jpeg to WhatsApp Image 2026-11-01 at 10.23.28 AM.jpeg
Open your project here:
https://dice-tank-porcupine.ngrok-free.dev


In [ ]:
!pip install fastapi uvicorn bcrypt pyjwt pyngrok -q
import os
os.makedirs('static', exist_ok=True)

In [ ]:
import os
from google.colab import userdata

# Load the required environment variables safely
required_secrets = ["JWT_SECRET", "EMAIL_ADDRESS", "EMAIL_PASSWORD"]
optional_secrets = ["ADMIN_USERNAME", "ADMIN_PASSWORD"]

for name in required_secrets:
    try:
        os.environ[name] = userdata.get(name)
    except Exception as e:
        print(f"Warning: Required secret '{name}' could not be loaded. Error: {e}")

for name in optional_secrets:
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        # Set a fallback or ignore if optional
        pass

print("Environment configuration completed successfully!")

Environment configuration completed successfully!


In [ ]:
%%writefile auth.py
"""
auth.py — Milestone 2, Member 1 (Step 3 + Step 4)
Login, signup, forgot/reset password (email OTP), JWT, profile API.

Secrets are read from environment variables only. In the Colab launcher cell:
    from google.colab import userdata
    import os
    for k in ("JWT_SECRET", "EMAIL_ADDRESS", "EMAIL_PASSWORD",
              "ADMIN_EMAIL", "ADMIN_PASSWORD"):
        os.environ[k] = userdata.get(k)
"""
import os
import re
import secrets
import smtplib
import sqlite3
import time
from contextlib import contextmanager
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage

import bcrypt
import jwt
from fastapi import APIRouter, HTTPException, Request
from pydantic import BaseModel

# Point this at the SAME sqlite file the FreightQuote app already uses
# (look for sqlite3.connect(...) in server.py / live_engine.py).
DB_PATH = os.environ.get("AUTH_DB", "freightquote.db")

PROJECT_NAME = "FreightQuote AI"
MAX_FAILS = 5
TOKEN_HOURS = 2
OTP_MINUTES = 5
DEFAULT_ROLE = "Customer"

router = APIRouter(prefix="/api/auth", tags=["auth"])
profile_router = APIRouter(prefix="/api/profile", tags=["profile"])


# ---------------------------------------------------------------- database
@contextmanager
def db():
    conn = sqlite3.connect(DB_PATH, timeout=10)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


def init_auth_db():
    """Create tables, then seed the first Admin from secrets (no hard-coded password)."""
    with db() as c:
        c.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id            INTEGER PRIMARY KEY AUTOINCREMENT,
                email         TEXT NOT NULL UNIQUE,
                username      TEXT NOT NULL UNIQUE,
                password_hash BLOB NOT NULL,
                role          TEXT NOT NULL DEFAULT 'Customer',
                locked        INTEGER NOT NULL DEFAULT 0,
                fail_count    INTEGER NOT NULL DEFAULT 0,
                customer_id   TEXT,
                created_at    TEXT NOT NULL,
                last_login    TEXT
            )""")
        c.execute("""
            CREATE TABLE IF NOT EXISTS password_resets (
                email    TEXT PRIMARY KEY,
                code_hash BLOB NOT NULL,
                expires  REAL NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0
            )""")
        has_admin = c.execute(
            "SELECT 1 FROM users WHERE role='Admin' LIMIT 1").fetchone()
        email = os.environ.get("ADMIN_EMAIL", "").strip().lower()
        pw = os.environ.get("ADMIN_PASSWORD", "")
        if not has_admin and email and valid_email(email) and valid_password(pw):
            c.execute(
                "INSERT INTO users (email, username, password_hash, role, created_at)"
                " VALUES (?,?,?,?,?)",
                (email, "admin", hash_pw(pw), "Admin", now_iso()))


# ---------------------------------------------------------------- validation
def valid_email(email: str) -> bool:
    """>=2 letters before @, >=2 chars between @ and the dot, >=2 letters after."""
    if email.count("@") != 1:
        return False
    local, domain = email.split("@")
    if len(re.findall(r"[A-Za-z]", local)) < 2:
        return False
    if not re.fullmatch(r"[A-Za-z0-9._%+-]+", local):
        return False
    return re.fullmatch(r"[A-Za-z0-9-]{2,}(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}", domain) is not None


def valid_password(pw: str) -> bool:
    return bool(len(pw) >= 8 and re.search(r"[A-Z]", pw) and re.search(r"[a-z]", pw)
                and re.search(r"\d", pw) and re.search(r"[^A-Za-z0-9]", pw))


def valid_username(u: str) -> bool:
    return re.fullmatch(r"[A-Za-z][A-Za-z0-9-]{2,29}", u) is not None


PASSWORD_RULE = ("Password needs 8+ characters with an uppercase letter, a lowercase letter, "
                 "a number and a special character.")
EMAIL_RULE = ("Email needs at least 2 letters before @, 2 characters between @ and the dot, "
              "and 2 letters after the dot.")
USERNAME_RULE = "Username must start with a letter and have 3-30 letters, numbers or hyphens."


# ---------------------------------------------------------------- hashing / JWT
def hash_pw(value: str) -> bytes:
    return bcrypt.hashpw(value.encode(), bcrypt.gensalt())


def check_pw(value: str, hashed) -> bool:
    try:
        if isinstance(hashed, str):
            hashed = hashed.encode()
        return bcrypt.checkpw(value.encode(), hashed)
    except (ValueError, TypeError):
        return False


_DUMMY_HASH = bcrypt.hashpw(b"dummy-password", bcrypt.gensalt())


def jwt_secret() -> str:
    s = os.environ.get("JWT_SECRET")
    if not s:
        raise RuntimeError("JWT_SECRET is not set. Add it to Colab Secrets and export it to os.environ.")
    return s


def make_token(user_id: int) -> str:
    now = datetime.now(timezone.utc)
    return jwt.encode({"sub": str(user_id), "iat": now,
                       "exp": now + timedelta(hours=TOKEN_HOURS)},
                      jwt_secret(), algorithm="HS256")


# ---------------------------------------------------------------- identity (replaces localuser.py)
def public_user(row) -> dict:
    """Never include password_hash."""
    return {"id": row["id"], "email": row["email"], "username": row["username"],
            "role": row["role"], "customer_id": row["customer_id"],
            "created_at": row["created_at"], "last_login": row["last_login"]}


def user_of(request: Request) -> dict:
    """Drop-in replacement for user_of(request) in server.py.
    Reads the Bearer token, re-checks the DB on EVERY request, so a deleted or
    locked account stops working immediately."""
    header = request.headers.get("Authorization", "")
    if not header.startswith("Bearer "):
        raise HTTPException(401, "Not signed in.")
    try:
        data = jwt.decode(header[7:], jwt_secret(), algorithms=["HS256"])
        uid = int(data["sub"])
    except (jwt.PyJWTError, KeyError, ValueError, TypeError):
        raise HTTPException(401, "Session expired. Please sign in again.")
    with db() as c:
        row = c.execute("SELECT * FROM users WHERE id=?", (uid,)).fetchone()
    if row is None or row["locked"]:
        raise HTTPException(401, "Account is not available.")
    return public_user(row)


# ---------------------------------------------------------------- request models
class SignupIn(BaseModel):
    email: str
    username: str
    password: str
    confirm: str


class LoginIn(BaseModel):
    identity: str   # email or username
    password: str


class ForgotIn(BaseModel):
    email: str


class ResetIn(BaseModel):
    email: str
    otp: str
    new_password: str
    confirm: str


class UsernameIn(BaseModel):
    username: str


class PasswordIn(BaseModel):
    current_password: str
    new_password: str
    confirm: str


# ---------------------------------------------------------------- email OTP
def send_otp(to_email: str, code: str):
    sender = os.environ.get("EMAIL_ADDRESS")
    app_pw = os.environ.get("EMAIL_PASSWORD")
    if not sender or not app_pw:
        raise RuntimeError("EMAIL_ADDRESS / EMAIL_PASSWORD are not set.")
    msg = EmailMessage()
    msg["Subject"] = f"{PROJECT_NAME} - Verification Code"
    msg["From"] = f"{PROJECT_NAME} <{sender}>"
    msg["To"] = to_email
    msg.set_content(f"Your verification code is {code}. It expires in {OTP_MINUTES} minutes.\n"
                    "If you did not request this, ignore this email.")
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as s:
        s.login(sender, app_pw)
        s.send_message(msg)


# ---------------------------------------------------------------- Step 3 routes
@router.post("/signup")
def signup(body: SignupIn):
    email = body.email.strip().lower()
    username = body.username.strip()
    if not all([email, username, body.password, body.confirm]):
        raise HTTPException(400, "All fields are mandatory.")
    if not valid_email(email):
        raise HTTPException(400, EMAIL_RULE)
    if not valid_username(username):
        raise HTTPException(400, USERNAME_RULE)
    if not valid_password(body.password):
        raise HTTPException(400, PASSWORD_RULE)
    if body.password != body.confirm:
        raise HTTPException(400, "Passwords do not match.")
    try:
        with db() as c:
            c.execute("INSERT INTO users (email, username, password_hash, role, created_at)"
                      " VALUES (?,?,?,?,?)",
                      (email, username, hash_pw(body.password), DEFAULT_ROLE, now_iso()))
    except sqlite3.IntegrityError:
        raise HTTPException(409, "That email or username is already registered.")
    return {"ok": True, "message": "Account created. Please sign in."}


@router.post("/login")
def login(body: LoginIn):
    identity = body.identity.strip()
    generic = HTTPException(401, "Invalid email or password.")   # same text for unknown / wrong
    if not identity or not body.password:
        raise HTTPException(400, "Enter your email and password.")
    with db() as c:
        row = c.execute("SELECT * FROM users WHERE email=? OR username=?",
                        (identity.lower(), identity)).fetchone()
        if row is None:
            check_pw(body.password, _DUMMY_HASH)     # keep timing similar
            raise generic
        if row["locked"]:
            raise HTTPException(403, "This account is locked. Contact an administrator.")
        if not check_pw(body.password, row["password_hash"]):
            fails = row["fail_count"] + 1
            c.execute("UPDATE users SET fail_count=?, locked=? WHERE id=?",
                      (fails, 1 if fails >= MAX_FAILS else 0, row["id"]))
            c.commit()   # keep the counter even though we raise below
            raise generic
        c.execute("UPDATE users SET fail_count=0, last_login=? WHERE id=?",
                  (now_iso(), row["id"]))
        user = c.execute("SELECT * FROM users WHERE id=?", (row["id"],)).fetchone()
    return {"token": make_token(user["id"]), "user": public_user(user)}


@router.post("/forgot")
def forgot(body: ForgotIn):
    email = body.email.strip().lower()
    generic = {"ok": True, "message": "If that email is registered, a code has been sent."}
    if not valid_email(email):
        raise HTTPException(400, EMAIL_RULE)
    with db() as c:
        row = c.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone()
        if not row:
            return generic                           # do not reveal whether the email exists
        code = f"{secrets.randbelow(1_000_000):06d}"
        c.execute("INSERT OR REPLACE INTO password_resets (email, code_hash, expires, attempts)"
                  " VALUES (?,?,?,0)", (email, hash_pw(code), time.time() + OTP_MINUTES * 60))
    try:
        send_otp(email, code)
    except Exception:
        raise HTTPException(503, "Could not send the email. Please try again later.")
    return generic


@router.post("/reset")
def reset(body: ResetIn):
    email = body.email.strip().lower()
    if not valid_password(body.new_password):
        raise HTTPException(400, PASSWORD_RULE)
    if body.new_password != body.confirm:
        raise HTTPException(400, "Passwords do not match.")
    bad = HTTPException(400, "Invalid or expired code.")
    with db() as c:
        r = c.execute("SELECT * FROM password_resets WHERE email=?", (email,)).fetchone()
        if r is None or r["expires"] < time.time() or r["attempts"] >= 5:
            raise bad
        if not check_pw(body.otp.strip(), r["code_hash"]):
            c.execute("UPDATE password_resets SET attempts=attempts+1 WHERE email=?", (email,))
            c.commit()
            raise bad
        c.execute("UPDATE users SET password_hash=? WHERE email=?",
                  (hash_pw(body.new_password), email))
        c.execute("DELETE FROM password_resets WHERE email=?", (email,))
    return {"ok": True, "message": "Password updated. Please sign in."}


# ---------------------------------------------------------------- Step 4 routes
@profile_router.get("")
def get_profile(request: Request):
    return user_of(request)


@profile_router.put("/username")
def change_username(body: UsernameIn, request: Request):
    me = user_of(request)
    username = body.username.strip()
    if not valid_username(username):
        raise HTTPException(400, USERNAME_RULE)
    try:
        with db() as c:
            c.execute("UPDATE users SET username=? WHERE id=?", (username, me["id"]))
    except sqlite3.IntegrityError:
        raise HTTPException(409, "That username is already taken.")
    return {"ok": True, "username": username}


@profile_router.post("/password")
def change_password(body: PasswordIn, request: Request):
    me = user_of(request)
    if not valid_password(body.new_password):
        raise HTTPException(400, PASSWORD_RULE)
    if body.new_password != body.confirm:
        raise HTTPException(400, "Passwords do not match.")
    with db() as c:
        row = c.execute("SELECT password_hash FROM users WHERE id=?", (me["id"],)).fetchone()
        if not check_pw(body.current_password, row["password_hash"]):
            raise HTTPException(400, "Current password is incorrect.")
        c.execute("UPDATE users SET password_hash=? WHERE id=?",
                  (hash_pw(body.new_password), me["id"]))
    return {"ok": True, "message": "Password changed."}

Writing auth.py
